In [1]:
import pandas as pd
import numpy as np

# Fleet model calibration notebook

The calibration is provided for year 2019.

1. Folder: `fleet_data_sources`
This folder contains the **source data** required for the calibration processes, including:

- List of modeled aircraft ICAO codes. 
- **BTS** traffic data for 2010, 2019 and 2025. It allows to cover a variety of aircraft which may be absent today of the US market, while still avoiding to manipulate a very heavy dataset. Identified with aircraft ICAO codes.
- **Seymour's simplified fuel model coefficients**.
- **Private TAA fleet data file**, not public. Every commercial passenger aircraft (no freight), identified with aircraft ICAO codes, with more than 10 active aircraft in 2019.
- **Airport datase**, provided by BTS as support table (@mention ISA). It is used to apply weights on the assignment (avoiding over representation of international flights in regional datasets). 
- **Aeroscope** total traffic for 2019. 
- **Not used here** : Eurocontrol traffic data for March, June, September and December 2019, not public. Provided with aircraft ICAO codes. https://ourairports.com database was used to apply weights on the Eurocontrol connections.


2. Folder: `fleet_calibrated_inputs_processed_pviry_repository`
This folder contains the **processed input data** using various functions and codes developed during **Paco Viry's thesis**. It includes:

- Sets of coefficients capturing **type-specific and age-related retirement propensity**.
- Coefficient capturing the **impact of aircraft age on utilization** : -x% per year.
- Quantitative mesure of the **impact of route distance on aircraft utilization**.

3. Folder: `fleet_calibrated_inputs_processed_here`
This folder contains the  **processed input data** built upon `data_sources` and `calibrated_inputs_processed_pviry_repository` in the notebook.
- Fleet structure in ?? per aircraft ICAO code and built year.
- Average number of seats per aircraft ICAO code (from BTS).
- E/ASK calculations per ICAO code.
- Productivity for age 0 aircraft (ASK/ac/y) calculated per ICAO code. Total estimated ASKs per ICAO type in ??. (including a correction factor to match the total ASKs calculated through the fleet structure, seats capacities and productivity mechanisms and ??? total market ASKs in ???).

## Fleet structure calculations

Etapes à réaliser de nuevo :
1) pas de transport ni de militaire (tri spécifique) pas de jet privé
2) nomenclatures des types, 
4) fusion des dates, nomenclature et tris sur le statut.
5) pas d avions produits à moins de 10 exemplaires.
6) garder les colonnes pertinentes
7) a posteriori, étudier les avions présents ici mais pas dans bts. les enlever à la main si nécessaire.

In [75]:
# def date_to_float_year(date):
#     if type(date)!= float :
#         start_of_year = pd.Timestamp(year=date.year, month=1, day=1)
#         start_of_next_year = pd.Timestamp(year=date.year + 1, month=1, day=1)
#         year_length = (start_of_next_year - start_of_year).days
#         days_since_start_of_year = (date - start_of_year).days
#         return date.year + days_since_start_of_year / year_length
#     else :
#         return date
    
# fleet_ini = pd.read_excel('fleet_data_sources/fleet_extract_250610.xlsx')
# fleet_ini = fleet_ini[fleet_ini['Airframe Body Type'].isin(['Narrowbody', 'Small Widebody', 'Turboprop', 'Regional Jet', 
#                                                             'Medium Widebody', 'Large Widebody'])]
# fleet_ini = fleet_ini[['Type', 'Model', 'Exp. Delivery', 'Status', 'Date', 'Year Built']] 


# taa_icao_names = pd.read_csv('fleet_data_sources/aircraft_codes_taa_icao.csv', sep=';')
# dict_taa_icao = dict(zip(taa_icao_names['Aircraft Type'], taa_icao_names['ICAO_Code']))
# fleet_ini['Aircraft Type'] = fleet_ini['Type'].astype(str) + fleet_ini['Model'].astype(str)
# fleet_ini['Aircraft Type'] = fleet_ini['Aircraft Type'].map(dict_taa_icao)
# fleet_ini = fleet_ini[~fleet_ini['Aircraft Type'].isna()]

# fleet_ini['Year Built']= fleet_ini['Year Built'].fillna(fleet_ini['Exp. Delivery'])
# fleet_ini = fleet_ini[fleet_ini['Year Built']!='12/30/1899']
# fleet_ini = fleet_ini[~fleet_ini['Year Built'].isna()]
# fleet_ini['Year Built'] = fleet_ini['Year Built'].apply(date_to_float_year)
# fleet_ini['Date'] = fleet_ini['Date'].apply(date_to_float_year)

# fleet_ini = fleet_ini[['Aircraft Type','Year Built', 'Status', 'Date']]
# fleet = fleet_ini[fleet_ini['Status'].isin(['Active', 'Stored', 'Written Off'])].reset_index(drop=True)
# fleet = fleet[~fleet['Aircraft Type'].isin(['DC95', 'DC92', 'L101', 'B742', 'DC91', 'AN32','IL86',
#        'L188', 'B743', 'DC10', 'MD81', 'DC93', 'B74S'])] #less than 10 active aircraft in 2019 - and no perspective
# fleet.to_csv('fleet_data_sources/fleet_data.csv')

In [76]:
fleet = pd.read_csv('fleet_data_sources/fleet_data.csv')
fleet.drop(columns=['Unnamed: 0'], inplace = True)
fleet['Year Built'] = fleet['Year Built'].astype(int)
y_min = fleet['Year Built'].min()

In [77]:
obs_year = 2019
fleet_selec = fleet[(fleet['Year Built']<=obs_year)&((fleet['Status'].isin(['Active', 'Stored']))|(fleet['Date']>obs_year))]
counts = fleet_selec.groupby(['Aircraft Type', 'Year Built']).size().unstack(fill_value=0)
counts.to_excel('fleet_calibrated_inputs_processed_here/agg_fleet_'+str(obs_year)+'.xlsx')

In [4]:
list_prov_taa = fleet['Aircraft Type'].unique()

## Energy per ASK calculations

### Seats and productivity estimation (BTS only)

In [5]:
# traffic_bts = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2010.csv')
# df = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2019.csv')
# traffic_bts = pd.concat([traffic_bts, df], ignore_index=True)
# df = pd.read_csv('fleet_data_sources/T_T100_SEGMENT_ALL_CARRIER_2025.csv')
# traffic_bts = pd.concat([traffic_bts, df], ignore_index=True)
# traffic_bts = traffic_bts[(traffic_bts['DEPARTURES_PERFORMED']>0)&(traffic_bts['SEATS']>0)&(traffic_bts['DISTANCE']>0)]

# bts_icao = pd.read_csv('fleet_data_sources/aircraft_codes_icao_joined.csv', sep=';')
# bts_icao_dict = dict(zip(bts_icao['Code'], bts_icao['ICAO_Code']))

# traffic_bts['Aircraft Type'] = traffic_bts['AIRCRAFT_TYPE'].map(bts_icao_dict)
# traffic_bts = traffic_bts[~traffic_bts['Aircraft Type'].isna()]
# traffic_bts = traffic_bts[['DEPARTURES_PERFORMED','SEATS','DISTANCE','ORIGIN_AIRPORT_ID','DEST_AIRPORT_ID','YEAR','Aircraft Type']]
# traffic_bts.to_csv('fleet_data_sources/traffic_data_bts_2010_2019_2025.csv')

In [81]:
traffic_bts = pd.read_csv('fleet_data_sources/traffic_data_bts_2010_2019_2025.csv')
traffic_bts.drop(columns=['Unnamed: 0'], inplace = True)

In [82]:
# weighting the connections
airports = pd.read_csv('fleet_data_sources/bts_airports.csv')[['AIRPORT_ID','AIRPORT_COUNTRY_NAME']]
us_airports = airports[airports['AIRPORT_COUNTRY_NAME']=='United States']
us_id = us_airports['AIRPORT_ID'].unique() #some inconsistencies regarding porto rico (minor)

traffic_bts['weight'] = 0
traffic_bts.loc[traffic_bts['ORIGIN_AIRPORT_ID'].isin(us_id), 'weight']+=0.5
traffic_bts.loc[traffic_bts['DEST_AIRPORT_ID'].isin(us_id), 'weight']+=0.5

In [83]:
list_prov_bts = traffic_bts['Aircraft Type'].unique()
icao_taa_in_bts = [a for a in list_prov_taa if a in list_prov_bts]
#operation caracteristics and number of seats will be associated to a "similar aircraft" present in bts.
icao_taa_not_in_bts = [a for a in list_prov_taa if a not in list_prov_bts] 

In [9]:
traffic_bts_s = traffic_bts[traffic_bts['Aircraft Type'].isin(icao_taa_in_bts)]

In [87]:
len(icao_taa_in_bts)

75

In [86]:
traffic_bts_s

,DEPARTURES_PERFORMED,SEATS,DISTANCE,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,YEAR,Aircraft Type,weight
0,1.0,1.0,25.0,12868,11844,2010,SH33,1.0
1,1.0,1.0,34.0,11844,13768,2010,SH33,1.0
4,1.0,1.0,44.0,11844,14474,2010,SH33,1.0
6,1.0,1.0,69.0,11844,12214,2010,SH33,1.0
7,1.0,1.0,78.0,15581,11007,2010,SH33,1.0
...,...,...,...,...,...,...,...,...
1274223,949.0,8568.0,59.0,10299,11555,2025,C208,1.0
1274224,1019.0,9468.0,59.0,10299,11555,2025,C208,1.0
1274225,1023.0,9469.0,59.0,11555,10299,2025,C208,1.0
1274226,1036.0,9297.0,59.0,10299,11555,2025,C208,1.0


In [157]:
traffic_bts_s

,DEPARTURES_PERFORMED,SEATS,DISTANCE,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,YEAR,Aircraft Type,weight
0,1.0,1.0,25.0,12868,11844,2010,SH33,1.0
1,1.0,1.0,34.0,11844,13768,2010,SH33,1.0
4,1.0,1.0,44.0,11844,14474,2010,SH33,1.0
6,1.0,1.0,69.0,11844,12214,2010,SH33,1.0
7,1.0,1.0,78.0,15581,11007,2010,SH33,1.0
...,...,...,...,...,...,...,...,...
1274223,949.0,8568.0,59.0,10299,11555,2025,C208,1.0
1274224,1019.0,9468.0,59.0,10299,11555,2025,C208,1.0
1274225,1023.0,9469.0,59.0,11555,10299,2025,C208,1.0
1274226,1036.0,9297.0,59.0,10299,11555,2025,C208,1.0


### Energy efficiency calculations

## Total ASKs calculations